In [1]:
from utils.spark_session import get_sparkSession 

In [2]:
spark = get_sparkSession("data init - Load dim_channel")

In [3]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [4]:
from delta import DeltaTable

In [5]:
row_count = spark.read.table("gold.dim_channel").count()

if row_count ==0:
    spark.sql(
    """INSERT INTO gold.dim_channel (channel_sk,
                                     channel_code,
                                     channel_name,
                                     channel_type,
                                     channel_desc,
                                     is_deleted,
                                     created_on,
                                     created_by,
                                     modified_on,
                                     modified_by)
         VALUES
           (1,'MKT','Marketplace','MARKETPLACE','Amazon / eBay',false,current_timestamp(),'DATA-INIT',current_timestamp(),'DATA_INIT'),
           (2,'RET','Retail','RETAIL','Physical retail stores',false,current_timestamp(),'DATA-INIT',current_timestamp(),'DATA_INIT'),
           (3,'DTC','Direct-to-customer','DIRECT','Brand-owned website',false,current_timestamp(),'DATA-INIT',current_timestamp(),'DATA_INIT'),
           (4,'B2B','Wholesale','B2B','Selling to distributors or other businesses',false,current_timestamp(),'DATA-INIT',current_timestamp(),'DATA_INIT')
    """)
    print("SPARK-APP: Rows Inserted")
else:
    print("SPARK-APP: Rows already exists")

SPARK-APP: Rows Inserted


In [6]:
spark.read.table("gold.dim_channel").show()

+----------+------------+------------------+------------+--------------------+----------+--------------------+----------+--------------------+-----------+
|channel_sk|channel_code|      channel_name|channel_type|        channel_desc|is_deleted|          created_on|created_by|         modified_on|modified_by|
+----------+------------+------------------+------------+--------------------+----------+--------------------+----------+--------------------+-----------+
|         4|         B2B|         Wholesale|         B2B|Selling to distri...|     false|2026-01-28 23:35:...| DATA-INIT|2026-01-28 23:35:...|  DATA_INIT|
|         3|         DTC|Direct-to-customer|      DIRECT| Brand-owned website|     false|2026-01-28 23:35:...| DATA-INIT|2026-01-28 23:35:...|  DATA_INIT|
|         1|         MKT|       Marketplace| MARKETPLACE|       Amazon / eBay|     false|2026-01-28 23:35:...| DATA-INIT|2026-01-28 23:35:...|  DATA_INIT|
|         2|         RET|            Retail|      RETAIL|Physical reta

In [7]:
dt = DeltaTable.forName(spark,"gold.dim_channel")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|1      |4            |
+-------+-------------+



In [ ]:
#Generating symlink manifest file

dt = DeltaTable.forName(spark, "gold.dim_channel")
dt.generate("symlink_format_manifest")

print("SPARK-APP: symlink manifest file generated")

In [8]:
spark.stop()